In [2]:
# ── Imports ───────────────────────────────────────────────────────
import pybaseball as pb
import pandas as pd
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
import pyro.infer.autoguide as autoguide
from pyro.optim import Adam
from sklearn.preprocessing import RobustScaler
from scipy.stats import spearmanr

# ── Season Config ─────────────────────────────────────────────────
PRED_SEASON = 2026          # season to predict
TEST_SEASON = PRED_SEASON - 1   # held-out season for backtesting

# ── Feature Sets ──────────────────────────────────────────────────
DECAY_FEATURES  = ['K%', 'BB%', 'GB%', 'CSW%']
RECENT_FEATURES = ['IP_per_G', 'start_IP_per_GS', 'relief_IP_per_G', 'HLDSV_per_G', 'RS/9']
LAMBDA = 0.5 ** (1/3)

SP_FEATURES = (
    [f'{f}_decay'  for f in ['K%', 'BB%', 'GB%', 'CSW%']] +
    [f'{f}_recent' for f in ['IP_per_G', 'start_IP_per_GS', 'RS/9']] +
    ['age', 'age_sq', 'experience', 'delta_IP_per_G']
)
RP_FEATURES = (
    [f'{f}_decay'  for f in ['K%', 'BB%', 'GB%', 'CSW%']] +
    [f'{f}_recent' for f in ['IP_per_G', 'relief_IP_per_G', 'HLDSV_per_G', 'RS/9']] +
    ['age', 'age_sq', 'experience', 'delta_IP_per_G']
)

SEASON_GAMES = {yr: (60 if yr == 2020 else 162) for yr in range(2019, TEST_SEASON + 1)}
COUNTING_COLS = ['IP', 'SO', 'H', 'ER', 'BB', 'HBP', 'WP', 'BK', 'W', 'L',
                 'BS', 'HLD', 'SV', 'Start-IP', 'Relief-IP', 'G', 'GS']

# ── Data Pipeline ────────────────────────────────────────────────
def pull_seasons(seasons):
    return pd.concat(
        [pb.pitching_stats(yr, yr, qual=1) for yr in seasons],
        ignore_index=True)

def normalize_season_length(df):
    df = df.copy()
    df['season_factor'] = df['Season'].map(SEASON_GAMES) / 162
    for col in COUNTING_COLS:
        df[col] = df[col] / df['season_factor']
    return df

def add_role_proxies(df):
    df['IP_per_G']        = df['IP'] / df['G']
    df['start_IP_per_GS'] = np.where(df['GS'] == 0, 0, df['Start-IP'] / df['GS'])
    df['relief_IP_per_G'] = df['Relief-IP'] / df['G']
    df['HLDSV_per_G']     = (df['HLD'] + df['SV']) / df['G']
    df['is_SP']           = (df['GS'] >= 10) & (df['start_IP_per_GS'] >= 5)
    return df

def add_fantasy_points(df):
    base = (df['IP']*2.49 + df['SO']*0.75 + df['H']*-0.75 + df['ER']*-2.75 +
            df['BB']*-0.5 + df['HBP']*-0.75 + df['WP']*-0.75 + df['BK']*-1.5 +
            df['W']*2.0 + df['L']*-1.0 + df['BS']*-2.0)
    rp_bonus = df['HLD']*2.25 + df['SV']*2.25
    df['fantasy_pts']        = np.where(df['is_SP'], base, base + rp_bonus)
    df['fantasy_pts_per_IP'] = df['fantasy_pts'] / np.clip(df['IP'], 10, None)
    return df

def build_rolling_windows(df):
    rows = []
    df = add_role_proxies(df)
    df = add_fantasy_points(df)
    for pid in df['IDfg'].unique():
        pdf = df[df['IDfg'] == pid].sort_values('Season')
        for target_season in pdf['Season'].values:
            past = pdf[pdf['Season'] < target_season].tail(3)
            if len(past) == 0:
                continue
            w = np.array([LAMBDA**(len(past)-1-j) for j in range(len(past))])
            w /= w.sum()
            row = {}
            for feat in DECAY_FEATURES:
                vals = past[feat].values
                ww = w.copy(); ww[np.isnan(vals)] = 0
                row[f'{feat}_decay'] = np.nan if ww.sum()==0 else (vals * (ww/ww.sum())).sum()
            for feat in RECENT_FEATURES:
                row[f'{feat}_recent'] = past[feat].iloc[-1]
            row['age']          = past['Age'].iloc[-1] + 1
            row['age_sq']       = row['age'] ** 2
            row['experience']   = len(past)
            row['is_SP']        = past.iloc[-1]['is_SP']
            row['delta_IP_per_G'] = np.diff(past['IP_per_G'].tail(2).values)[0] if len(past)>1 else 0
            tgt = pdf[pdf['Season']==target_season].iloc[0]
            sf  = SEASON_GAMES.get(target_season, 162) / 162
            row['target_fantasy_pts_per_IP'] = tgt['fantasy_pts_per_IP']
            row['target_IP']  = tgt['IP'] * sf
            row['pitcher']    = pid
            row['season']     = target_season
            rows.append(row)
    return pd.DataFrame(rows)

def build_inference_rows(df, role='SP'):
    recent = (['IP_per_G', 'start_IP_per_GS', 'RS/9'] if role=='SP'
              else ['IP_per_G', 'relief_IP_per_G', 'HLDSV_per_G', 'RS/9'])
    rows = []
    df = add_role_proxies(df)
    df = add_fantasy_points(df)
    for pid in df['IDfg'].unique():
        pdf = df[df['IDfg']==pid].sort_values('Season')
        mr  = pdf.iloc[-1]
        if role=='SP' and not mr['is_SP']: continue
        if role=='RP' and mr['is_SP']:     continue
        past = pdf.tail(3)
        if len(past) == 0: continue
        w = np.array([LAMBDA**(len(past)-1-j) for j in range(len(past))])
        w /= w.sum()
        row = {}
        for feat in DECAY_FEATURES:
            vals = past[feat].values
            ww = w.copy(); ww[np.isnan(vals)] = 0
            row[f'{feat}_decay'] = np.nan if ww.sum()==0 else (vals*(ww/ww.sum())).sum()
        for feat in recent:
            row[f'{feat}_recent'] = past[feat].iloc[-1]
        row['name']          = mr['Name']
        row['age']           = past['Age'].iloc[-1] + 1
        row['age_sq']        = row['age'] ** 2
        row['experience']    = len(past)
        row['is_SP']         = mr['is_SP']
        row['delta_IP_per_G'] = np.diff(past['IP_per_G'].tail(2).values)[0] if len(past)>1 else 0
        row['pitcher']       = pid
        rows.append(row)
    return pd.DataFrame(rows)

# ── Splits & Tensors ─────────────────────────────────────────────
def train_test_split(windows_df, test_season=None):
    """Pass test_season=None to train on all data (for new-season inference)."""
    if test_season is None:
        return windows_df, None
    return (windows_df[windows_df['season'] < test_season],
            windows_df[windows_df['season'] == test_season])

def split_by_role(windows_df):
    return (windows_df[windows_df['is_SP']],
            windows_df[~windows_df['is_SP']])

def prepare_tensors(role_train, role_df, features, scaler=None):
    """Returns (X, pitcher_idx, y_pts, y_IP, scaler, encoder, filtered_role_df)."""
    encoder  = {pid: idx for idx, pid in enumerate(role_train['pitcher'].unique())}
    role_df  = (role_df
                .dropna(subset=features)
                [lambda d: d['pitcher'].isin(encoder)]
                .reset_index(drop=True))
    if scaler is None:
        scaler = RobustScaler()
        X = torch.tensor(scaler.fit_transform(role_df[features]), dtype=torch.float32)
    else:
        X = torch.tensor(scaler.transform(role_df[features]), dtype=torch.float32)
    pitcher_idx = torch.tensor(role_df['pitcher'].map(encoder).values, dtype=torch.long)
    y_pts = torch.tensor(role_df['target_fantasy_pts_per_IP'].values, dtype=torch.float32)
    y_IP  = torch.tensor(role_df['target_IP'].values, dtype=torch.float32)
    return X, pitcher_idx, y_pts, y_IP, scaler, encoder, role_df

# ── Model ────────────────────────────────────────────────────────
def model(X, pitcher_idx, n_pitchers, y_pts_per_IP=None, y_IP=None, role='SP'):
    n_features     = X.shape[1]
    ip_prior_mean  = 5.0 if role=='SP' else 3.7
    ip_sigma_prior = 50.0 if role=='SP' else 20.0

    alpha_pts   = pyro.sample('alpha_pts',  dist.Normal(0, 1))
    alpha_IP    = pyro.sample('alpha_IP',   dist.Normal(ip_prior_mean, 1))
    beta_pts    = pyro.sample('beta_pts',   dist.Normal(0,1).expand([n_features]).to_event(1))
    beta_IP     = pyro.sample('beta_IP',    dist.Normal(0,1).expand([n_features]).to_event(1))
    sigma_u_pts = pyro.sample('sigma_u_pts', dist.HalfNormal(1))
    sigma_u_IP  = pyro.sample('sigma_u_IP',  dist.HalfNormal(1))

    with pyro.plate('pitchers', n_pitchers):
        u_pts     = pyro.sample('u_pts',     dist.Normal(0, sigma_u_pts))
        u_IP      = pyro.sample('u_IP',      dist.Normal(0, sigma_u_IP))
        sigma_pts = pyro.sample('sigma_pts', dist.HalfNormal(1))
        sigma_IP  = pyro.sample('sigma_IP',  dist.HalfNormal(ip_sigma_prior))

    with pyro.plate('obs', X.shape[0]):
        mu_pts = alpha_pts + u_pts[pitcher_idx] + (X * beta_pts).sum(-1)
        mu_IP  = torch.exp(
            (alpha_IP + u_IP[pitcher_idx] + (X * beta_IP).sum(-1)).clamp(-6, 6)
        ).clamp(min=1e-4)
        sig_IP = sigma_IP[pitcher_idx].clamp(min=1e-4)
        pyro.sample('y_pts', dist.Normal(mu_pts, sigma_pts[pitcher_idx]),    obs=y_pts_per_IP)
        pyro.sample('y_IP',  dist.Gamma(mu_IP**2/sig_IP**2, mu_IP/sig_IP**2), obs=y_IP)

def sp_model(X, pitcher_idx, n_pitchers, y_pts_per_IP=None, y_IP=None):
    return model(X, pitcher_idx, n_pitchers, y_pts_per_IP, y_IP, role='SP')

def rp_model(X, pitcher_idx, n_pitchers, y_pts_per_IP=None, y_IP=None):
    return model(X, pitcher_idx, n_pitchers, y_pts_per_IP, y_IP, role='RP')

# ── Inference ────────────────────────────────────────────────────
def run_inference(model_fn, X, pitcher_idx, n_pitchers,
                  y_pts, y_IP, n_steps=3000, lr=0.01):
    torch.manual_seed(42)
    pyro.set_rng_seed(42)
    pyro.clear_param_store()
    guide = autoguide.AutoDiagonalNormal(model_fn)
    svi   = SVI(model_fn, guide, Adam({'lr': lr}), loss=Trace_ELBO())
    losses = []
    for step in range(n_steps):
        loss = svi.step(X, pitcher_idx, n_pitchers, y_pts, y_IP)
        losses.append(loss)
        if step % 500 == 0:
            print(f'  step {step:4d}  loss {loss:.1f}')
    return guide, losses

# ── Evaluation ───────────────────────────────────────────────────
def evaluate(guide, model_fn, X_test, pitcher_idx_test, n_pitchers, test_df):
    predictive = Predictive(model_fn, guide=guide, num_samples=1000)
    samples    = predictive(X_test, pitcher_idx_test, n_pitchers)
    total      = (samples['y_pts'] * samples['y_IP']).detach().numpy()
    mean_pts   = total.mean(axis=0)
    std_pts    = total.std(axis=0)
    p10        = np.percentile(total, 10, axis=0)
    p90        = np.percentile(total, 90, axis=0)
    score      = mean_pts / np.clip(std_pts, 10, None)
    actual     = (test_df['target_fantasy_pts_per_IP'] * test_df['target_IP']).values
    rankings = pd.DataFrame({
        'pitcher': test_df['pitcher'].values,
        'mean_pts': mean_pts, 'std_pts': std_pts,
        'score': score, 'p10': p10, 'p90': p90, 'actual_pts': actual
    }).sort_values('score', ascending=False)
    metrics = pd.DataFrame({
        'spearman_mean':  [spearmanr(mean_pts, actual).correlation],
        'spearman_score': [spearmanr(score,    actual).correlation],
        'coverage_80pct': [((actual >= p10) & (actual <= p90)).mean()]
    })
    return rankings, metrics, samples

# ── New-Season Prediction ─────────────────────────────────────────
def risk_adjusted_score(mean_pts, std_pts, alpha=0.5, floor=10):
    """
    alpha=0: pure mean ranking (upside-weighted)
    alpha=0.5: balanced (default)
    alpha=1: pure Sharpe ratio (floor-based)
    """
    return mean_pts / np.clip(std_pts, floor, None) ** alpha

def predict_new_season(guide, model_fn, inference_df, features,
                       scaler, encoder, n_pitchers, alpha = 0.5, floor = 10):
    """n_pitchers = len(encoder) from training. The +1 slot (unobserved prior) handles unknowns."""
    df = inference_df.copy()
    df['is_known'] = df['pitcher'].map(lambda p: p in encoder)
    df[features]   = df[features].fillna(0)
    df['pitcher_idx'] = df['pitcher'].map(encoder).fillna(n_pitchers).astype(int)

    X           = torch.tensor(scaler.transform(df[features]), dtype=torch.float32)
    pitcher_idx = torch.tensor(df['pitcher_idx'].values, dtype=torch.long)

    predictive = Predictive(model_fn, guide=guide, num_samples=1000)
    samples    = predictive(X, pitcher_idx, n_pitchers + 1)
    total      = (samples['y_pts'] * samples['y_IP']).detach().numpy()

    mean_pts = total.mean(axis=0)
    std_pts  = total.std(axis=0)
    p10      = np.percentile(total, 10, axis=0)
    p90      = np.percentile(total, 90, axis=0)
    score    = risk_adjusted_score(mean_pts, std_pts, alpha=alpha, floor=floor)

    return pd.DataFrame({
        'name':     df['name'].values,
        'age':      df['age'].values,
        'mean_pts': mean_pts.round(1),
        'std_pts':  std_pts.round(1),
        'p10':      p10.round(1),
        'p90':      p90.round(1),
        'score':    score.round(3),
        'is_known': df['is_known'].values,
    }).sort_values('score', ascending=False).reset_index(drop=True)

/opt/miniconda3/envs/dugout/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pull_seasons(range(2019, PRED_SEASON))
df = normalize_season_length(df)
windows_df = build_rolling_windows(df)
windows_df['relief_IP_per_G_recent'] = windows_df['relief_IP_per_G_recent'].fillna(0)

train, test = train_test_split(windows_df, test_season=TEST_SEASON)
sp_train, sp_test = split_by_role(train)[0], split_by_role(test)[0]
rp_train, rp_test = split_by_role(train)[1], split_by_role(test)[1]

print(f"Seasons loaded: {sorted(windows_df['season'].unique())}")
print(f"Backtest split: train < {TEST_SEASON}, test = {TEST_SEASON}")
print(f"SP — train {len(sp_train)}, test {len(sp_test)}")
print(f"RP — train {len(rp_train)}, test {len(rp_test)}")

Seasons loaded: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Backtest split: train < 2025, test = 2025
SP — train 562, test 136
RP — train 2617, test 558


# Starting Pitchers

In [4]:
# ── Backtest: train < TEST_SEASON, evaluate on TEST_SEASON ───────
X_sp_train, pitcher_idx_sp_train, y_pts_sp_train, y_IP_sp_train, \
    scaler_sp, sp_encoder, sp_train_df = \
    prepare_tensors(sp_train, sp_train, SP_FEATURES)

X_sp_test, pitcher_idx_sp_test, y_pts_sp_test, y_IP_sp_test, \
    _, _, sp_test_df = \
    prepare_tensors(sp_train, sp_test, SP_FEATURES, scaler=scaler_sp)

n_pitchers_sp = len(sp_encoder)
print(f"SP: {len(X_sp_train)} train obs, {len(X_sp_test)} test obs, {n_pitchers_sp} pitchers")

sp_guide, sp_losses = run_inference(
    sp_model, X_sp_train, pitcher_idx_sp_train, n_pitchers_sp,
    y_pts_sp_train, y_IP_sp_train)

sp_rankings, sp_metrics, sp_samples = evaluate(
    sp_guide, sp_model, X_sp_test, pitcher_idx_sp_test, n_pitchers_sp, sp_test_df)

print(sp_metrics.to_string(index=False))

SP: 562 train obs, 101 test obs, 256 pitchers
  step    0  loss 61809.2
  step  500  loss 4194.7
  step 1000  loss 3995.2
  step 1500  loss 3963.5
  step 2000  loss 3939.9
  step 2500  loss 3943.6
 spearman_mean  spearman_score  coverage_80pct
      0.447373        0.326523        0.792079


In [5]:
# ── SP Diagnostics ────────────────────────────────────────────────
total    = (sp_samples['y_pts'] * sp_samples['y_IP']).detach().numpy()
actual   = sp_test_df['target_fantasy_pts_per_IP'].values * sp_test_df['target_IP'].values

print(f"Predicted  mean={total.mean():.1f}  std={total.std(axis=0).mean():.1f}")
print(f"Actual     mean={actual.mean():.1f}  std={actual.std():.1f}")
print(f"80% interval width: {(np.percentile(total,90,axis=0) - np.percentile(total,10,axis=0)).mean():.1f}")

# baseline
print(f"\nBaseline Spearman (K%_decay):  {spearmanr(-sp_test_df['K%_decay'], actual).correlation:.3f}")
print(f"Model Spearman (mean_pts):     {spearmanr(total.mean(axis=0), actual).correlation:.3f}")

# feature importance
locs = pyro.get_param_store()['AutoDiagonalNormal.loc'].detach().numpy()
n_f  = len(SP_FEATURES)
print(pd.DataFrame({
    'feature': SP_FEATURES,
    'beta_pts': locs[2:2+n_f].round(3),
    'beta_IP':  locs[2+n_f:2+2*n_f].round(3)
}).sort_values('beta_pts', key=abs, ascending=False).to_string(index=False))

Predicted  mean=138.2  std=116.0
Actual     mean=135.0  std=103.4
80% interval width: 246.1

Baseline Spearman (K%_decay):  -0.386
Model Spearman (mean_pts):     0.447
               feature  beta_pts  beta_IP
              K%_decay     0.520    0.110
                age_sq    -0.396   -0.042
                   age     0.337   -0.025
             GB%_decay     0.105    0.136
            CSW%_decay    -0.097    0.001
             BB%_decay    -0.086   -0.045
            experience    -0.073    0.562
       IP_per_G_recent    -0.051    0.099
start_IP_per_GS_recent     0.042   -0.087
        delta_IP_per_G     0.034    0.029
           RS/9_recent     0.002    0.030


# SP: New Season Predictions

In [6]:
# ── Train on ALL seasons, predict PRED_SEASON ─────────────────────
print(f"Training SP on all data through {PRED_SEASON - 1}...")
train_all, _ = train_test_split(windows_df, test_season=None)
sp_train_all  = split_by_role(train_all)[0]

X_sp_all, pitcher_idx_sp_all, y_pts_sp_all, y_IP_sp_all, \
    scaler_sp_final, sp_encoder_final, _ = \
    prepare_tensors(sp_train_all, sp_train_all, SP_FEATURES)

n_pitchers_sp_all = len(sp_encoder_final)
# train with n_pitchers + 1: last slot is unobserved (prior = N(0, sigma_u))
sp_guide_final, sp_losses_final = run_inference(
    sp_model, X_sp_all, pitcher_idx_sp_all, n_pitchers_sp_all + 1,
    y_pts_sp_all, y_IP_sp_all)

sp_inference       = build_inference_rows(df, role='SP')
sp_rankings_new    = predict_new_season(
    sp_guide_final, sp_model, sp_inference,
    SP_FEATURES, scaler_sp_final, sp_encoder_final, n_pitchers_sp_all, alpha = 0.2)

print(f"\n=== Top 20 SP for {PRED_SEASON} ===")
print(sp_rankings_new.head(20).to_string(index=False))

Training SP on all data through 2025...
  step    0  loss 57797.0
  step  500  loss 5272.2
  step 1000  loss 4968.2
  step 1500  loss 4892.2
  step 2000  loss 4869.7
  step 2500  loss 4841.9

=== Top 20 SP for 2026 ===
              name  age   mean_pts    std_pts        p10        p90      score  is_known
        Logan Webb   29 288.299988  86.800003 188.199997 394.700012 118.083000      True
      Zack Wheeler   36 271.600006 103.300003 152.300003 408.600006 107.417000      True
       Paul Skenes   24 290.399994 155.000000 120.300003 476.799988 105.925003      True
      Tarik Skubal   29 294.299988 173.899994  99.000000 508.700012 104.885002      True
        Kris Bubic   28 303.399994 215.100006  80.800003 574.500000 103.616997      True
     Logan Gilbert   29 257.299988 132.000000 121.099998 421.500000  96.894997      True
      Hunter Brown   27 254.199997 139.300003  99.199997 416.600006  94.712997      True
          Joe Ryan   30 232.800003 101.099998 123.099998 354.600006  

# Relief Pitchers

In [7]:
# ── Backtest: train < TEST_SEASON, evaluate on TEST_SEASON ───────
X_rp_train, pitcher_idx_rp_train, y_pts_rp_train, y_IP_rp_train, \
    scaler_rp, rp_encoder, rp_train_df = \
    prepare_tensors(rp_train, rp_train, RP_FEATURES)

X_rp_test, pitcher_idx_rp_test, y_pts_rp_test, y_IP_rp_test, \
    _, _, rp_test_df = \
    prepare_tensors(rp_train, rp_test, RP_FEATURES, scaler=scaler_rp)

n_pitchers_rp = len(rp_encoder)
print(f"RP: {len(X_rp_train)} train obs, {len(X_rp_test)} test obs, {n_pitchers_rp} pitchers")

rp_guide, rp_losses = run_inference(
    rp_model, X_rp_train, pitcher_idx_rp_train, n_pitchers_rp,
    y_pts_rp_train, y_IP_rp_train)

rp_rankings, rp_metrics, rp_samples = evaluate(
    rp_guide, rp_model, X_rp_test, pitcher_idx_rp_test, n_pitchers_rp, rp_test_df)

print(rp_metrics.to_string(index=False))

RP: 2617 train obs, 407 test obs, 1135 pitchers
  step    0  loss 708459.3
  step  500  loss 20633.7
  step 1000  loss 18744.7
  step 1500  loss 18693.6
  step 2000  loss 18071.1
  step 2500  loss 18021.1
 spearman_mean  spearman_score  coverage_80pct
      0.418204        0.375267        0.683047


In [8]:
# ── RP Diagnostics ────────────────────────────────────────────────
total  = (rp_samples['y_pts'] * rp_samples['y_IP']).detach().numpy()
actual = rp_test_df['target_fantasy_pts_per_IP'].values * rp_test_df['target_IP'].values

print(f"Predicted  mean={total.mean():.1f}  std={total.std(axis=0).mean():.1f}")
print(f"Actual     mean={actual.mean():.1f}  std={actual.std():.1f}")
print(f"80% interval width: {(np.percentile(total,90,axis=0) - np.percentile(total,10,axis=0)).mean():.1f}")

print(f"\nBaseline Spearman (K%_decay):  {spearmanr(-rp_test_df['K%_decay'], actual).correlation:.3f}")
print(f"Model Spearman (mean_pts):     {spearmanr(total.mean(axis=0), actual).correlation:.3f}")

locs = pyro.get_param_store()['AutoDiagonalNormal.loc'].detach().numpy()
n_f  = len(RP_FEATURES)
print(pd.DataFrame({
    'feature': RP_FEATURES,
    'beta_pts': locs[2:2+n_f].round(3),
    'beta_IP':  locs[2+n_f:2+2*n_f].round(3)
}).sort_values('beta_pts', key=abs, ascending=False).to_string(index=False))

Predicted  mean=49.7  std=62.3
Actual     mean=61.7  std=72.4
80% interval width: 123.1

Baseline Spearman (K%_decay):  -0.361
Model Spearman (mean_pts):     0.418
               feature  beta_pts  beta_IP
    HLDSV_per_G_recent     0.337    0.002
              K%_decay     0.328    0.134
                   age    -0.128    0.330
            experience     0.105    0.458
                age_sq     0.074   -0.423
       IP_per_G_recent    -0.048    0.092
            CSW%_decay     0.041    0.050
relief_IP_per_G_recent    -0.017   -0.036
             BB%_decay    -0.012   -0.033
           RS/9_recent     0.002   -0.006
        delta_IP_per_G     0.002   -0.003
             GB%_decay    -0.000    0.035


# RP: New Season Predictions

In [9]:
# ── Train on ALL seasons, predict PRED_SEASON ─────────────────────
print(f"Training RP on all data through {PRED_SEASON - 1}...")
rp_train_all = split_by_role(train_all)[1]

X_rp_all, pitcher_idx_rp_all, y_pts_rp_all, y_IP_rp_all, \
    scaler_rp_final, rp_encoder_final, _ = \
    prepare_tensors(rp_train_all, rp_train_all, RP_FEATURES)

n_pitchers_rp_all = len(rp_encoder_final)
rp_guide_final, rp_losses_final = run_inference(
    rp_model, X_rp_all, pitcher_idx_rp_all, n_pitchers_rp_all + 1,
    y_pts_rp_all, y_IP_rp_all)

rp_inference    = build_inference_rows(df, role='RP')
rp_rankings_new = predict_new_season(
    rp_guide_final, rp_model, rp_inference,
    RP_FEATURES, scaler_rp_final, rp_encoder_final, n_pitchers_rp_all, alpha = 0.2)

print(f"\n=== Top 20 RP for {PRED_SEASON} ===")
print(rp_rankings_new.head(20).to_string(index=False))

Training RP on all data through 2025...
  step    0  loss 992799.6
  step  500  loss 25553.7
  step 1000  loss 23032.7
  step 1500  loss 22217.1
  step 2000  loss 22930.6
  step 2500  loss 21433.0

=== Top 20 RP for 2026 ===
           name  age   mean_pts    std_pts       p10        p90     score  is_known
   Andres Munoz   27 160.500000  85.900002 65.800003 264.100006 65.864998      True
     Edwin Diaz   32 152.000000  85.500000 58.900002 261.799988 62.439999      True
    Jhoan Duran   28 141.300003  60.000000 69.099998 218.800003 62.310001      True
   Mason Miller   27 163.600006 135.300003 28.400000 328.299988 61.300999      True
    Griffin Jax   31 150.399994 109.699997 39.000000 288.600006 58.798000      True
  Kyle Finnegan   34 124.599998  43.200001 73.199997 182.100006 58.688000      True
   Tanner Scott   31 137.000000  81.900002 46.000000 244.500000 56.756001      True
     Phil Maton   33 121.300003  53.299999 59.000000 187.600006 54.764999      True
      Will Vest   3

In [15]:
sp_rankings_new.head(300).to_csv('./outputs/sp_rankings.csv', index=False)
rp_rankings_new.head(500).to_csv('./outputs/rp_rankings.csv', index=False)